<div style="background:linear-gradient(135deg,#312E81,#7E22CE,#BE123C);border-radius:24px;padding:44px 48px;color:white">
<div style="font-size:14px;letter-spacing:0.12em">BRANDCAST · DATA PREPARATION</div>
<h1 style="margin:10px 0;font-size:36px">RACOI 분석용 프로그램 테이블 생성</h1>
<p style="margin:0;font-size:17px">원본을 바꾸지 않고 프로그램 식별 기준을 분리합니다.</p>
</div>

<div style="background:#F5F3FF;border-left:6px solid #7C3AED;border-radius:14px;padding:22px 26px">
<h2 style="margin:0;color:#312E81">핵심 질문</h2>
<p style="margin:10px 0 0;color:#111827;font-size:17px">주간·월간 원본에서 같은 프로그램을 안전하게 식별하기 위해 어떤 이름·채널 기준으로 분석용 프로그램 1행을 만들 수 있는가?</p>
</div>

## 1. 목적과 범위

`raw_internet_response`와 `raw_total_response`에서 프로그램·채널 조합별 `analysis_program`을 만든다. raw 테이블은 수정하지 않는다. 댓글 API의 `pjseq`는 이름·채널과 자동 연결할 근거가 없으므로 이 단계에서는 비워 둔다.

분석용 프로그램 1행은 **정규화한 프로그램명 + 정규화한 채널명** 조합이다. 프로그램명이 같아도 채널이 다르면 별도 행으로 보존한다.

In [1]:
# 목적: 비밀값을 표시하지 않고 MariaDB와 JupySQL 연결을 준비한다.
# 입력: 로컬 .env의 MariaDB 환경변수.
# 출력 단위: MariaDB 연결 1개. 기간: 원본 보유 전체 기간. NULL·0 처리: 값을 조회·변경하지 않는다.
import os, sys
from pathlib import Path
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == '노트북' else Path.cwd()))
from 수집.원본_적재 import connect

connection = connect()
cursor = connection.cursor()
cursor.execute('SELECT VERSION()')
print(f'MariaDB 연결 성공: {cursor.fetchone()[0]}')
database_url = URL.create('mariadb+mariadbconnector', username=os.environ['MARIADB_USER'], password=os.environ['MARIADB_PASSWORD'], host=os.environ['MARIADB_HOST'], port=int(os.environ['MARIADB_PORT']), database=os.environ['MARIADB_DATABASE'])
engine = create_engine(database_url)
%load_ext sql
%config SqlMagic.named_parameters = 'disabled'
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql engine

MariaDB 연결 성공: 11.8.3-MariaDB


## 2. 처리 전 확인과 적용 기준

원본에는 `* 나 혼자 산다`처럼 앞 기호·공백이 붙은 이름이 있다. 앞쪽 별표와 공백을 제거하고 연속 공백을 하나로 줄이며 비교용으로 소문자로 바꾼다. 화면에 보여 줄 프로그램명·채널명은 원본에서 처음 관측한 표기를 유지한다.

서로 다른 이름을 같은 이름으로 바꾸거나, 다른 채널의 프로그램을 합치지 않는다. 따라서 의미가 다른 프로그램을 자동 병합하지 않는다.

In [2]:
%%sql
-- 목적: 프로그램명·채널 정규화 전에 원본별 조합 규모와 빈 이름을 확인한다.
-- 입력: raw_internet_response, raw_total_response.
-- 출력 단위: 원본 유형 1행. 기간: 각 raw 테이블의 전체 적재 기간. NULL·0 처리: 빈 이름은 0으로 바꾸지 않고 별도 건수로 센다.
SELECT '주간 인터넷반응' AS 원본_유형, COUNT(*) AS 행_수, COUNT(DISTINCT CONCAT_WS('|', program_name, channel_name)) AS 원본_프로그램채널_조합_수, SUM(program_name IS NULL OR TRIM(program_name) = '') AS 빈_프로그램명_수, SUM(channel_name IS NULL OR TRIM(channel_name) = '') AS 빈_채널명_수
FROM raw_internet_response
UNION ALL
SELECT '월간 통합반응', COUNT(*), COUNT(DISTINCT CONCAT_WS('|', program_name, channel_name)), SUM(program_name IS NULL OR TRIM(program_name) = ''), SUM(channel_name IS NULL OR TRIM(channel_name) = '')
FROM raw_total_response;

원본_유형,행_수,원본_프로그램채널_조합_수,빈_프로그램명_수,빈_채널명_수
주간 인터넷반응,8484,487,0,0
월간 통합반응,2797,484,0,0


## 3. `analysis_program` 생성

분석 테이블은 raw에서 다시 만들 수 있도록 재생성한다. 주간과 월간 기간 코드는 비교하지 않고 `first_weekly_period`·`last_weekly_period`, `first_monthly_period`·`last_monthly_period`로 나누어 보존한다. 날짜 변환과 지표 전처리는 이후 주간·월간 분석 테이블에서 각각 수행한다.

In [3]:
%%sql
-- 목적: raw를 수정하지 않고 분석용 프로그램 테이블 구조를 재생성한다.
-- 입력: 없음. 출력 단위: 프로그램·정규화 채널 조합 1행. 기간: raw 전체 기간. NULL·0 처리: pjseq는 근거가 없으면 NULL로 남긴다.
-- 재실행 시 기존 분석 자식 테이블 때문에 raw를 건드리지 못하는 일을 막기 위해, 이 세션에서만 외래키 검사를 잠시 끈다.
SET FOREIGN_KEY_CHECKS = 0;
DROP TABLE IF EXISTS analysis_program;
SET FOREIGN_KEY_CHECKS = 1;
CREATE TABLE analysis_program (
 program_id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT, pjseq BIGINT NULL, program_name VARCHAR(255) NOT NULL, normalized_program_name VARCHAR(255) NOT NULL, channel_name VARCHAR(255) NOT NULL, normalized_channel_name VARCHAR(255) NOT NULL, first_weekly_period VARCHAR(16) NULL, last_weekly_period VARCHAR(16) NULL, first_monthly_period VARCHAR(16) NULL, last_monthly_period VARCHAR(16) NULL, has_weekly_data BOOLEAN NOT NULL, has_monthly_data BOOLEAN NOT NULL, has_comment_data BOOLEAN NOT NULL, mapping_status VARCHAR(40) NOT NULL, PRIMARY KEY (program_id), UNIQUE KEY uq_analysis_program_normalized (normalized_program_name, normalized_channel_name)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

++
||
++
++

In [4]:
%%sql
-- 목적: 주간·월간 raw에서 정규화 프로그램명·채널별 프로그램 1행을 만든다.
-- 입력: raw_internet_response, raw_total_response.
-- 출력 단위: 정규화 프로그램명·정규화 채널명 조합 1행. 기간: raw 전체 기간. NULL·0 처리: 빈 이름·채널은 대체하지 않고 제외하며 pjseq·댓글 여부는 NULL·FALSE로 둔다.
INSERT INTO analysis_program (pjseq, program_name, normalized_program_name, channel_name, normalized_channel_name, first_weekly_period, last_weekly_period, first_monthly_period, last_monthly_period, has_weekly_data, has_monthly_data, has_comment_data, mapping_status)
WITH source_program AS (
 SELECT program_name, channel_name, period_code, 'weekly' AS source_grain FROM raw_internet_response
 UNION ALL
 SELECT program_name, channel_name, period_code, 'monthly' AS source_grain FROM raw_total_response
), normalized AS (
 -- 앞 기호·여러 공백은 표기 차이만 제거하며 채널이 다르면 GROUP BY가 자동 병합을 막는다.
 SELECT program_name, channel_name, period_code, source_grain, LOWER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(program_name, '^[*[:space:]]+', ''), '[[:space:]]+', ' '))) AS normalized_program_name, LOWER(TRIM(REGEXP_REPLACE(channel_name, '[[:space:]]+', ' '))) AS normalized_channel_name
 FROM source_program
 WHERE program_name IS NOT NULL AND TRIM(program_name) <> '' AND channel_name IS NOT NULL AND TRIM(channel_name) <> ''
)
SELECT NULL, SUBSTRING_INDEX(GROUP_CONCAT(program_name ORDER BY period_code, program_name SEPARATOR '|||'), '|||', 1), normalized_program_name, SUBSTRING_INDEX(GROUP_CONCAT(channel_name ORDER BY period_code, channel_name SEPARATOR '|||'), '|||', 1), normalized_channel_name, MIN(CASE WHEN source_grain = 'weekly' THEN period_code END), MAX(CASE WHEN source_grain = 'weekly' THEN period_code END), MIN(CASE WHEN source_grain = 'monthly' THEN period_code END), MAX(CASE WHEN source_grain = 'monthly' THEN period_code END), MAX(source_grain = 'weekly'), MAX(source_grain = 'monthly'), FALSE, 'name_channel_normalized'
FROM normalized
GROUP BY normalized_program_name, normalized_channel_name;

++
||
++
++

## 4. 생성 결과와 해석

`has_comment_data = 0`은 댓글이 없다는 뜻이 아니라, 현재 댓글 표본의 `pjseq`를 프로그램명·채널에 자동 연결하지 않았다는 뜻이다. 확인된 후보의 매핑만 이후에 추가한다.

In [5]:
%%sql
-- 목적: analysis_program 생성 후 원천 보유 상태별 프로그램 수와 대표 표본을 확인한다.
-- 입력: analysis_program. 출력 단위: 프로그램 1행. 기간: raw 전체 관측 기간. NULL·0 처리: pjseq NULL·댓글 매핑 0은 근거 미확보 상태이며 댓글 0건으로 해석하지 않는다.
SELECT program_id, pjseq, program_name, normalized_program_name, channel_name, normalized_channel_name, first_weekly_period, last_weekly_period, first_monthly_period, last_monthly_period, has_weekly_data, has_monthly_data, has_comment_data, mapping_status
FROM analysis_program
ORDER BY program_id
LIMIT 10;

program_id,pjseq,program_name,normalized_program_name,channel_name,normalized_channel_name,first_weekly_period,last_weekly_period,first_monthly_period,last_monthly_period,has_weekly_data,has_monthly_data,has_comment_data,mapping_status
1,None,[확장판] 사이비 헌터,[확장판] 사이비 헌터,MBC,mbc,202621,202622,202605,202605,1,1,0,name_channel_normalized
2,None,* 100분 토론,100분 토론,MBC,mbc,202547,202646,202511,202610,1,1,0,name_channel_normalized
3,None,100일의 거짓말,100일의 거짓말,tvN,tvn,202641,202647,202610,202610,1,1,0,name_channel_normalized
4,None,100일의 거짓말 : 비밀문서,100일의 거짓말 : 비밀문서,tvN,tvn,202640,202640,202609,202610,1,1,0,name_channel_normalized
5,None,1등들,1등들,MBC,mbc,202607,202616,202602,202604,1,1,0,name_channel_normalized
6,None,1박2일 시즌4,1박2일 시즌4,KBS2,kbs2,202547,202647,202511,202610,1,1,0,name_channel_normalized
7,None,2025 KBS 가요대축제 글로벌 페스티벌,2025 kbs 가요대축제 글로벌 페스티벌,KBS2,kbs2,202551,202551,202512,202512,1,1,0,name_channel_normalized
8,None,2025 KBS 연기대상,2025 kbs 연기대상,KBS2,kbs2,202601,202601,202512,202601,1,1,0,name_channel_normalized
9,None,2025 KBS 연예대상,2025 kbs 연예대상,KBS2,kbs2,202551,202551,202512,202512,1,1,0,name_channel_normalized
10,None,2025 MBC 가요대제전,2025 mbc 가요대제전,MBC,mbc,202601,202601,202512,202601,1,1,0,name_channel_normalized


## 5. 분석용 주간 테이블 생성

`analysis_program_weekly`는 프로그램·완료 주차 1행이다. 같은 프로그램·주차에 원본 payload가 둘 이상이면 어떤 값을 대표로 삼을지 추측하지 않고 `conflicting_source_version` 상태로 남기며 수치 열은 NULL로 둔다. 원본 수치는 합산하지 않는다.

In [6]:
%%sql
-- 목적: 프로그램·완료 주차 1행의 분석용 주간 테이블 구조를 재생성한다.
-- 입력: 없음. 출력 단위: program_id·week_code 1행. 기간: raw 주간 전체. NULL·0 처리: 충돌·변환 실패 수치는 NULL로 보존한다.
DROP TABLE IF EXISTS analysis_program_weekly;
CREATE TABLE analysis_program_weekly (
 program_id BIGINT UNSIGNED NOT NULL, week_code VARCHAR(16) NOT NULL, week_start_date DATE NOT NULL, week_end_date DATE NOT NULL,
 post_count DECIMAL(20,6) NULL, comment_count DECIMAL(20,6) NULL, video_view_count DECIMAL(20,6) NULL, news_count DECIMAL(20,6) NULL, media_video_count DECIMAL(20,6) NULL,
 post_share DECIMAL(20,6) NULL, comment_share DECIMAL(20,6) NULL, video_view_share DECIMAL(20,6) NULL, news_share DECIMAL(20,6) NULL, media_video_share DECIMAL(20,6) NULL,
 household_viewer_count DECIMAL(20,6) NULL, viewer_2049_count DECIMAL(20,6) NULL, household_rating DECIMAL(20,6) NULL, rating_2049 DECIMAL(20,6) NULL,
 source_checksum CHAR(64) NULL, quality_status VARCHAR(40) NOT NULL, PRIMARY KEY (program_id, week_code),
 CONSTRAINT fk_analysis_weekly_program FOREIGN KEY (program_id) REFERENCES analysis_program(program_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

++
||
++
++

In [7]:
%%sql
-- 목적: 주간 raw의 단일 내용 원본만 숫자로 변환해 프로그램·주차 행을 만든다.
-- 입력: raw_internet_response, analysis_program. 출력 단위: program_id·week_code 1행. 기간: raw 주간 전체. NULL·0 처리: 내용 충돌·숫자 변환 실패는 수치를 NULL로 두고 상태를 기록한다.
INSERT INTO analysis_program_weekly (program_id, week_code, week_start_date, week_end_date, post_count, comment_count, video_view_count, news_count, media_video_count, post_share, comment_share, video_view_share, news_share, media_video_share, household_viewer_count, viewer_2049_count, household_rating, rating_2049, source_checksum, quality_status)
WITH normalized AS (
 SELECT r.*, LOWER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(r.program_name, '^[*[:space:]]+', ''), '[[:space:]]+', ' '))) AS normalized_program_name, LOWER(TRIM(REGEXP_REPLACE(r.channel_name, '[[:space:]]+', ' '))) AS normalized_channel_name
 FROM raw_internet_response r
 WHERE r.program_name IS NOT NULL AND TRIM(r.program_name) <> '' AND r.channel_name IS NOT NULL AND TRIM(r.channel_name) <> ''
), grouped AS (
 SELECT normalized_program_name, normalized_channel_name, period_code, COUNT(DISTINCT raw_payload_json) AS payload_variant_count, MIN(source_checksum) AS source_checksum,
        MIN(viewer_post_value) AS viewer_post_value, MIN(viewer_comment_value) AS viewer_comment_value, MIN(viewer_video_view_value) AS viewer_video_view_value, MIN(media_news_value) AS media_news_value, MIN(media_video_value) AS media_video_value,
        MIN(viewer_post_share_value) AS viewer_post_share_value, MIN(viewer_comment_share_value) AS viewer_comment_share_value, MIN(viewer_video_view_share_value) AS viewer_video_view_share_value, MIN(media_news_share_value) AS media_news_share_value, MIN(media_video_share_value) AS media_video_share_value,
        MIN(household_viewer_value) AS household_viewer_value, MIN(viewer_2049_value) AS viewer_2049_value, MIN(household_rating_value) AS household_rating_value, MIN(rating_2049_value) AS rating_2049_value
 FROM normalized GROUP BY normalized_program_name, normalized_channel_name, period_code
)
SELECT p.program_id, g.period_code, STR_TO_DATE(CONCAT(g.period_code, ' Monday'), '%x%v %W'), DATE_ADD(STR_TO_DATE(CONCAT(g.period_code, ' Monday'), '%x%v %W'), INTERVAL 6 DAY),
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_post_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_comment_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_video_view_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.media_news_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.media_video_value, ',', '') AS DECIMAL(20,6)) END,
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_post_share_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_comment_share_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_video_view_share_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.media_news_share_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.media_video_share_value, ',', '') AS DECIMAL(20,6)) END,
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.household_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_2049_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.household_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.rating_2049_value, ',', '') AS DECIMAL(20,6)) END,
 CASE WHEN g.payload_variant_count = 1 THEN g.source_checksum END, CASE WHEN g.payload_variant_count > 1 THEN 'conflicting_source_version' ELSE 'valid' END
FROM grouped g JOIN analysis_program p ON p.normalized_program_name = g.normalized_program_name AND p.normalized_channel_name = g.normalized_channel_name;

++
||
++
++

In [8]:
%%sql
-- 목적: 주간 분석 테이블의 유효·충돌 행 수와 관측 기간을 확인한다.
-- 입력: analysis_program_weekly. 출력 단위: quality_status 1행. 기간: 주간 전체. NULL·0 처리: 충돌 행의 NULL 수치는 0으로 세지 않는다.
SELECT quality_status, COUNT(*) AS 프로그램주차_행수, COUNT(DISTINCT program_id) AS 프로그램_수, MIN(week_code) AS 시작_주차, MAX(week_code) AS 종료_주차
FROM analysis_program_weekly GROUP BY quality_status ORDER BY quality_status;

quality_status,프로그램주차_행수,프로그램_수,시작_주차,종료_주차
conflicting_source_version,154,154,202647,202647
valid,7868,481,202547,202646


## 6. 분석용 월간 테이블 생성

`analysis_program_monthly`는 프로그램·완료 월 1행이다. 월간 값은 주간 값으로 나누거나 주간 행과 조인하지 않는다. 동일 프로그램·월의 내용 충돌은 주간과 같은 방식으로 대표값을 고르지 않고 상태로 남긴다.

In [9]:
%%sql
-- 목적: 프로그램·완료 월 1행의 분석용 월간 테이블 구조를 재생성한다.
-- 입력: 없음. 출력 단위: program_id·month_code 1행. 기간: raw 월간 전체. NULL·0 처리: 원본 단위가 불명확한 플랫폼 값도 숫자 원문은 보존하고 상태를 기록한다.
DROP TABLE IF EXISTS analysis_program_monthly;
CREATE TABLE analysis_program_monthly (
 program_id BIGINT UNSIGNED NOT NULL, month_code VARCHAR(16) NOT NULL, month_start_date DATE NOT NULL, month_end_date DATE NOT NULL,
 post_count DECIMAL(20,6) NULL, comment_count DECIMAL(20,6) NULL, video_view_count DECIMAL(20,6) NULL, news_count DECIMAL(20,6) NULL, media_video_count DECIMAL(20,6) NULL,
 tv_household_viewer_count DECIMAL(20,6) NULL, tv_individual_viewer_count DECIMAL(20,6) NULL, tv_male_viewer_count DECIMAL(20,6) NULL, tv_female_viewer_count DECIMAL(20,6) NULL, tv_age10_viewer_count DECIMAL(20,6) NULL, tv_age20_viewer_count DECIMAL(20,6) NULL, tv_age30_viewer_count DECIMAL(20,6) NULL, tv_age40_viewer_count DECIMAL(20,6) NULL, tv_age50_viewer_count DECIMAL(20,6) NULL,
 tv_household_rating DECIMAL(20,6) NULL, tv_individual_rating DECIMAL(20,6) NULL, tv_male_rating DECIMAL(20,6) NULL, tv_female_rating DECIMAL(20,6) NULL, tv_age10_rating DECIMAL(20,6) NULL, tv_age20_rating DECIMAL(20,6) NULL, tv_age30_rating DECIMAL(20,6) NULL, tv_age40_rating DECIMAL(20,6) NULL, tv_age50_rating DECIMAL(20,6) NULL,
 tv_vod_value DECIMAL(20,6) NULL, pc_live_value DECIMAL(20,6) NULL, pc_vod_value DECIMAL(20,6) NULL, mobile_live_value DECIMAL(20,6) NULL, mobile_vod_value DECIMAL(20,6) NULL, platform_unit_status VARCHAR(40) NOT NULL, source_checksum CHAR(64) NULL, quality_status VARCHAR(40) NOT NULL, PRIMARY KEY (program_id, month_code), CONSTRAINT fk_analysis_monthly_program FOREIGN KEY (program_id) REFERENCES analysis_program(program_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

++
||
++
++

In [10]:
%%sql
-- 목적: 월간 raw의 단일 내용 원본만 프로그램·월 행으로 만든다.
-- 입력: raw_total_response, analysis_program. 출력 단위: program_id·month_code 1행. 기간: raw 월간 전체. NULL·0 처리: 내용 충돌은 수치를 NULL로 두고, 플랫폼 지표 단위는 확인 전 상태로 기록한다.
INSERT INTO analysis_program_monthly (program_id, month_code, month_start_date, month_end_date, post_count, comment_count, video_view_count, news_count, media_video_count, tv_household_viewer_count, tv_individual_viewer_count, tv_male_viewer_count, tv_female_viewer_count, tv_age10_viewer_count, tv_age20_viewer_count, tv_age30_viewer_count, tv_age40_viewer_count, tv_age50_viewer_count, tv_household_rating, tv_individual_rating, tv_male_rating, tv_female_rating, tv_age10_rating, tv_age20_rating, tv_age30_rating, tv_age40_rating, tv_age50_rating, tv_vod_value, pc_live_value, pc_vod_value, mobile_live_value, mobile_vod_value, platform_unit_status, source_checksum, quality_status)
WITH normalized AS (
 SELECT r.*, LOWER(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(r.program_name, '^[*[:space:]]+', ''), '[[:space:]]+', ' '))) AS normalized_program_name, LOWER(TRIM(REGEXP_REPLACE(r.channel_name, '[[:space:]]+', ' '))) AS normalized_channel_name
 FROM raw_total_response r WHERE r.program_name IS NOT NULL AND TRIM(r.program_name) <> '' AND r.channel_name IS NOT NULL AND TRIM(r.channel_name) <> ''
), grouped AS (
 SELECT normalized_program_name, normalized_channel_name, period_code, COUNT(DISTINCT raw_payload_json) AS payload_variant_count, MIN(source_checksum) AS source_checksum,
 MIN(viewer_post_value) AS viewer_post_value, MIN(viewer_comment_value) AS viewer_comment_value, MIN(viewer_video_view_value) AS viewer_video_view_value, MIN(media_news_value) AS media_news_value, MIN(media_video_value) AS media_video_value,
 MIN(tv_household_viewer_value) AS tv_household_viewer_value, MIN(tv_individual_viewer_value) AS tv_individual_viewer_value, MIN(tv_male_viewer_value) AS tv_male_viewer_value, MIN(tv_female_viewer_value) AS tv_female_viewer_value, MIN(tv_age10_viewer_value) AS tv_age10_viewer_value, MIN(tv_age20_viewer_value) AS tv_age20_viewer_value, MIN(tv_age30_viewer_value) AS tv_age30_viewer_value, MIN(tv_age40_viewer_value) AS tv_age40_viewer_value, MIN(tv_age50_viewer_value) AS tv_age50_viewer_value,
 MIN(tv_household_rating_value) AS tv_household_rating_value, MIN(tv_individual_rating_value) AS tv_individual_rating_value, MIN(tv_male_rating_value) AS tv_male_rating_value, MIN(tv_female_rating_value) AS tv_female_rating_value, MIN(tv_age10_rating_value) AS tv_age10_rating_value, MIN(tv_age20_rating_value) AS tv_age20_rating_value, MIN(tv_age30_rating_value) AS tv_age30_rating_value, MIN(tv_age40_rating_value) AS tv_age40_rating_value, MIN(tv_age50_rating_value) AS tv_age50_rating_value, MIN(tv_vod_value) AS tv_vod_value, MIN(pc_live_value) AS pc_live_value, MIN(pc_vod_value) AS pc_vod_value, MIN(mobile_live_value) AS mobile_live_value, MIN(mobile_vod_value) AS mobile_vod_value
 FROM normalized GROUP BY normalized_program_name, normalized_channel_name, period_code
)
SELECT p.program_id, g.period_code, STR_TO_DATE(CONCAT(g.period_code, '01'), '%Y%m%d'), LAST_DAY(STR_TO_DATE(CONCAT(g.period_code, '01'), '%Y%m%d')),
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_post_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_comment_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.viewer_video_view_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.media_news_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.media_video_value, ',', '') AS DECIMAL(20,6)) END,
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_household_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_individual_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_male_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_female_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age10_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age20_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age30_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age40_viewer_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age50_viewer_value, ',', '') AS DECIMAL(20,6)) END,
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_household_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_individual_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_male_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_female_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age10_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age20_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age30_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age40_rating_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_age50_rating_value, ',', '') AS DECIMAL(20,6)) END,
 CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.tv_vod_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.pc_live_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.pc_vod_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.mobile_live_value, ',', '') AS DECIMAL(20,6)) END, CASE WHEN g.payload_variant_count = 1 THEN CAST(REPLACE(g.mobile_vod_value, ',', '') AS DECIMAL(20,6)) END,
 'unit_unverified', CASE WHEN g.payload_variant_count = 1 THEN g.source_checksum END, CASE WHEN g.payload_variant_count > 1 THEN 'conflicting_source_version' ELSE 'valid' END
FROM grouped g JOIN analysis_program p ON p.normalized_program_name = g.normalized_program_name AND p.normalized_channel_name = g.normalized_channel_name;

++
||
++
++

In [11]:
%%sql
-- 목적: 월간 분석 테이블의 유효·충돌 행 수와 관측 기간을 확인한다.
-- 입력: analysis_program_monthly. 출력 단위: quality_status 1행. 기간: 월간 전체. NULL·0 처리: 충돌 행의 NULL 수치는 0으로 세지 않는다.
SELECT quality_status, COUNT(*) AS 프로그램월_행수, COUNT(DISTINCT program_id) AS 프로그램_수, MIN(month_code) AS 시작_월, MAX(month_code) AS 종료_월
FROM analysis_program_monthly GROUP BY quality_status ORDER BY quality_status;

quality_status,프로그램월_행수,프로그램_수,시작_월,종료_월
conflicting_source_version,186,186,202610,202610
valid,2050,453,202511,202610


## 7. 댓글 키워드·감성·연관어 분석 테이블 생성

댓글 원천은 현재 프로그램 ID `11487`의 시범 수집분이다. 따라서 세 테이블은 구조와 품질 검증용으로 만들며, 전체 프로그램 후보의 비교 지표로 사용하지 않는다. `program_id`는 자동 추정하지 않고 NULL로 둔다.

In [12]:
%%sql
-- 목적: 댓글 키워드·감성·연관어 원천별 분석 테이블을 raw를 수정하지 않고 재생성한다.
-- 입력: 없음. 출력 단위: 원본 댓글 배열 객체 1행. 기간: 댓글 시범 수집 기간. NULL·0 처리: 미매핑 program_id는 NULL이며 0이나 임의 프로그램으로 바꾸지 않는다.
DROP TABLE IF EXISTS analysis_keyword_weekly;
DROP TABLE IF EXISTS analysis_sentiment_weekly;
DROP TABLE IF EXISTS analysis_keyword_edge;
CREATE TABLE analysis_keyword_weekly (keyword_row_id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT, program_id BIGINT UNSIGNED NULL, pjseq BIGINT NOT NULL, week_code VARCHAR(16) NOT NULL, dataset_name VARCHAR(80) NOT NULL, rank_value INT NULL, keyword_text VARCHAR(500) NULL, frequency_value DECIMAL(20,6) NULL, sentiment_type_code VARCHAR(20) NULL, source_checksum CHAR(64) NOT NULL, quality_status VARCHAR(40) NOT NULL, PRIMARY KEY(keyword_row_id), UNIQUE KEY uq_keyword_source(source_checksum,dataset_name,keyword_text,rank_value)) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
CREATE TABLE analysis_sentiment_weekly (sentiment_row_id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT, program_id BIGINT UNSIGNED NULL, pjseq BIGINT NOT NULL, request_week_code VARCHAR(16) NOT NULL, response_week_code VARCHAR(16) NOT NULL, dataset_name VARCHAR(80) NOT NULL, positive_ratio DECIMAL(20,12) NULL, negative_ratio DECIMAL(20,12) NULL, source_checksum CHAR(64) NOT NULL, quality_status VARCHAR(40) NOT NULL, PRIMARY KEY(sentiment_row_id), UNIQUE KEY uq_sentiment_source(source_checksum,dataset_name,response_week_code)) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
CREATE TABLE analysis_keyword_edge (edge_row_id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT, program_id BIGINT UNSIGNED NULL, pjseq BIGINT NOT NULL, week_code VARCHAR(16) NOT NULL, dataset_name VARCHAR(80) NOT NULL, rank_value INT NULL, source_keyword VARCHAR(500) NULL, target_keyword VARCHAR(500) NULL, frequency_value DECIMAL(20,6) NULL, source_checksum CHAR(64) NOT NULL, quality_status VARCHAR(40) NOT NULL, PRIMARY KEY(edge_row_id), UNIQUE KEY uq_edge_source(source_checksum,dataset_name,source_keyword,target_keyword,rank_value)) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

++
||
++
++

In [13]:
%%sql
-- 목적: 댓글 raw 객체를 키워드·감성·연관어 단위로 변환 없이 분리 적재한다.
-- 입력: raw_comment_keyword, raw_comment_sentiment, raw_comment_network. 출력 단위: 각 raw 배열 객체 1행. 기간: 댓글 시범 수집 기간. NULL·0 처리: 숫자 변환 실패는 NULL과 quality_status로 기록하며, 감성 비율 합계가 1이 아니면 별도 상태로 남긴다.
INSERT INTO analysis_keyword_weekly (program_id,pjseq,week_code,dataset_name,rank_value,keyword_text,frequency_value,sentiment_type_code,source_checksum,quality_status)
SELECT NULL, request_pjseq, response_nweek, dataset_name, rank_value, keyword_text, CASE WHEN frequency_value REGEXP '^[0-9]+([.][0-9]+)?$' THEN CAST(frequency_value AS DECIMAL(20,6)) END, sentiment_type_code, source_checksum, CASE WHEN keyword_text IS NULL OR TRIM(keyword_text) = '' THEN 'missing_keyword' WHEN frequency_value NOT REGEXP '^[0-9]+([.][0-9]+)?$' THEN 'numeric_conversion_failed' ELSE 'valid' END
FROM raw_comment_keyword;
INSERT INTO analysis_sentiment_weekly (program_id,pjseq,request_week_code,response_week_code,dataset_name,positive_ratio,negative_ratio,source_checksum,quality_status)
SELECT NULL, request_pjseq, request_nweek, response_nweek, dataset_name, CASE WHEN positive_score_value REGEXP '^[0-9]+([.][0-9]+)?$' THEN CAST(positive_score_value AS DECIMAL(20,12)) END, CASE WHEN negative_score_value REGEXP '^[0-9]+([.][0-9]+)?$' THEN CAST(negative_score_value AS DECIMAL(20,12)) END, source_checksum, CASE WHEN positive_score_value NOT REGEXP '^[0-9]+([.][0-9]+)?$' OR negative_score_value NOT REGEXP '^[0-9]+([.][0-9]+)?$' THEN 'numeric_conversion_failed' WHEN ABS(CAST(positive_score_value AS DECIMAL(20,12)) + CAST(negative_score_value AS DECIMAL(20,12)) - 1) > 0.000001 THEN 'ratio_sum_unexpected' ELSE 'valid' END
FROM raw_comment_sentiment;
INSERT INTO analysis_keyword_edge (program_id,pjseq,week_code,dataset_name,rank_value,source_keyword,target_keyword,frequency_value,source_checksum,quality_status)
SELECT NULL, request_pjseq, request_nweek, dataset_name, rank_value, source_keyword, target_keyword, CASE WHEN frequency_value REGEXP '^[0-9]+([.][0-9]+)?$' THEN CAST(frequency_value AS DECIMAL(20,6)) END, source_checksum, CASE WHEN source_keyword IS NULL OR TRIM(source_keyword) = '' THEN 'missing_source_keyword' WHEN frequency_value IS NOT NULL AND frequency_value NOT REGEXP '^[0-9]+([.][0-9]+)?$' THEN 'numeric_conversion_failed' ELSE 'valid' END
FROM raw_comment_network;

++
||
++
++

In [14]:
%%sql
-- 목적: 댓글 분석 테이블의 유형·상태별 행 수와 시범 수집 식별자를 확인한다.
-- 입력: analysis_keyword_weekly, analysis_sentiment_weekly, analysis_keyword_edge. 출력 단위: 분석 테이블·자료유형·상태 1행. 기간: 댓글 시범 수집 기간. NULL·0 처리: program_id NULL은 미매핑이며 댓글 0건으로 해석하지 않는다.
SELECT '키워드' AS 테이블, dataset_name AS 자료유형, quality_status AS 상태, COUNT(*) AS 행_수, MIN(pjseq) AS 프로그램_ID, MIN(week_code) AS 시작_주차, MAX(week_code) AS 종료_주차 FROM analysis_keyword_weekly GROUP BY dataset_name, quality_status
UNION ALL
SELECT '감성', dataset_name, quality_status, COUNT(*), MIN(pjseq), MIN(response_week_code), MAX(response_week_code) FROM analysis_sentiment_weekly GROUP BY dataset_name, quality_status
UNION ALL
SELECT '연관어', dataset_name, quality_status, COUNT(*), MIN(pjseq), MIN(week_code), MAX(week_code) FROM analysis_keyword_edge GROUP BY dataset_name, quality_status
ORDER BY 테이블, 자료유형;

테이블,자료유형,상태,행_수,프로그램_ID,시작_주차,종료_주차
감성,sentiment_distribution,valid,1,11487,202646,202646
감성,sentiment_trend,valid,14,11487,202633,202646
연관어,network_edges,valid,42,11487,202646,202646
연관어,network_nodes,valid,7,11487,202646,202646
연관어,network_rank,valid,10,11487,202646,202646
키워드,comments_rank,valid,20,11487,202646,202646
키워드,positive_comments_rank,valid,20,11487,202646,202646
키워드,wordcloud,valid,800,11487,202646,202646


## 8. 현재 분석 가능 범위

주간·월간 반응은 원천 단위를 유지한 채 분석용 테이블로 재생성할 수 있다. 동일 프로그램·기간의 서로 다른 원본 버전은 자동으로 선택하지 않고 충돌 상태로 남겼다. 댓글 테이블은 프로그램 ID 11487의 시범 수집분이므로 스키마·품질 검증에는 사용하지만 전체 후보의 감성·키워드 비교에는 사용하지 않는다.